In [6]:
from __future__ import annotations

import re

LINES = [
    "IPP: 123456 ; Nom: DUPONT ; Prénom: Alice",
    "Nom - Martin  Prenom:  Bob  IPP=987654",
    "IPP  135790 Nom:  O'Neil   Prenom - Shaun",
    "Prénom: Chloé ; Nom: Durand",
    "IPP: ABCDEF ; Nom: Test ; Prénom: X",
    "Random noise without fields",
    "IPP: 000001;Nom:LEBRUN;Prénom:Jean",
]

EXPECTED = [
    {"ipp": "123456", "nom": "DUPONT", "prenom": "Alice"},
    {"ipp": "987654", "nom": "Martin", "prenom": "Bob"},
    {"ipp": "135790", "nom": "O'Neil", "prenom": "Shaun"},
    None,
    None,
    None,
    {"ipp": "000001", "nom": "LEBRUN", "prenom": "Jean"},
]


def _norm_spaces(s: str) -> str:
    return " ".join(s.strip().split())


RX_IPP = re.compile(
    r"\bIPP\b\s*[:=\-]?\s*(?P<ipp>\d{6})\b"
    r"(?=\s*(?:;|\bIPP\b|\bNom\b|\bPr[ée]nom\b|$))",
    re.IGNORECASE,
)

RX_NOM = re.compile(
    r"\bNom\b\s*[:=\-]?\s*(?P<nom>[A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ'\- ]*[A-Za-zÀ-ÖØ-öø-ÿ])"
    r"(?=\s*(?:;|\bIPP\b|\bNom\b|\bPr[ée]nom\b|$))",
    re.IGNORECASE,
)

RX_PRENOM = re.compile(
    r"\bPr[ée]nom\b\s*[:=\-]?\s*(?P<prenom>[A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ'\- ]*[A-Za-zÀ-ÖØ-öø-ÿ])"
    r"(?=\s*(?:;|\bIPP\b|\bNom\b|\bPr[ée]nom\b|$))",
    re.IGNORECASE,
)


def extract_identity(line: str) -> dict[str, str] | None:
    m_ipp = RX_IPP.search(line)
    m_nom = RX_NOM.search(line)
    m_prenom = RX_PRENOM.search(line)

    if not (m_ipp and m_nom and m_prenom):
        return None

    return {
        "ipp": m_ipp.group("ipp"),
        "nom": _norm_spaces(m_nom.group("nom")),
        "prenom": _norm_spaces(m_prenom.group("prenom")),
    }


if __name__ == "__main__":
    results = [extract_identity(s) for s in LINES]

    for i, (line, got, exp) in enumerate(zip(LINES, results, EXPECTED), start=1):
        ok = got == exp
        print(f"{i}. OK={ok} | got={got} | expected={exp} | line={line}")

    assert results == EXPECTED
    print("All tests passed.")

1. OK=True | got={'ipp': '123456', 'nom': 'DUPONT', 'prenom': 'Alice'} | expected={'ipp': '123456', 'nom': 'DUPONT', 'prenom': 'Alice'} | line=IPP: 123456 ; Nom: DUPONT ; Prénom: Alice
2. OK=True | got={'ipp': '987654', 'nom': 'Martin', 'prenom': 'Bob'} | expected={'ipp': '987654', 'nom': 'Martin', 'prenom': 'Bob'} | line=Nom - Martin  Prenom:  Bob  IPP=987654
3. OK=False | got={'ipp': '135790', 'nom': "O'Neil Prenom - Shaun", 'prenom': 'Shaun'} | expected={'ipp': '135790', 'nom': "O'Neil", 'prenom': 'Shaun'} | line=IPP  135790 Nom:  O'Neil   Prenom - Shaun
4. OK=True | got=None | expected=None | line=Prénom: Chloé ; Nom: Durand
5. OK=True | got=None | expected=None | line=IPP: ABCDEF ; Nom: Test ; Prénom: X
6. OK=True | got=None | expected=None | line=Random noise without fields
7. OK=True | got={'ipp': '000001', 'nom': 'LEBRUN', 'prenom': 'Jean'} | expected={'ipp': '000001', 'nom': 'LEBRUN', 'prenom': 'Jean'} | line=IPP: 000001;Nom:LEBRUN;Prénom:Jean


AssertionError: 